# DR-ASPP-DRN: DME Head Fine-Tuning Pipeline

End-to-end notebook for IRDID dataset preprocessing, model training, and evaluation.

## 1. Environment Setup & Imports

In [ ]:
# Install dependencies (uncomment on Colab / Kaggle)
# !pip install -r requirements.txt

import os, json, logging, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(message)s")

print("TensorFlow", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))


## 2. Configuration

In [ ]:
import yaml

with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

DATASET_PATH   = cfg["dataset"]["path"]
BACKBONE_WEIGHTS = cfg["model"]["backbone_weights"]  # None or path
BATCH_SIZE     = cfg["training"]["batch_size"]
EPOCHS         = cfg["training"]["epochs"]
LR             = cfg["training"]["learning_rate"]
OUTPUT_DIR     = cfg["output"]["dir"]
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Dataset : {DATASET_PATH}")
print(f"Backbone: {BACKBONE_WEIGHTS}")
print(f"Output  : {OUTPUT_DIR}")


## 3. Load Dataset with Preprocessing

In [ ]:
from dataset_loader import IDRiDDataset

train_obj = IDRiDDataset(
    DATASET_PATH, split="train", augment=True, batch_size=BATCH_SIZE
)
val_obj = IDRiDDataset(
    DATASET_PATH, split="val", augment=False, batch_size=BATCH_SIZE
)

train_ds = train_obj.get_dataset()
val_ds   = val_obj.get_dataset()
class_weights = train_obj.get_class_weights()

print(f"Train samples : {len(train_obj)}")
print(f"Val   samples : {len(val_obj)}")
print(f"Class weights : {class_weights}")


## 4. Visualise Sample Batch

In [ ]:
train_obj.visualize_batch(n_samples=4)


## 5. Build and Compile Model

In [ ]:
from model import build_model_dme_tuning

model = build_model_dme_tuning(backbone_weights_path=BACKBONE_WEIGHTS)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=LR),
    loss={
        "dr_output": "mse",
        "dme_risk":  "categorical_crossentropy",
    },
    loss_weights={"dr_output": 0.0, "dme_risk": 1.0},
    metrics={"dr_output": [], "dme_risk": ["accuracy"]},
)

# Verify trainable layers
trainable = [(l.name, l.count_params())
             for l in model.layers if l.trainable and l.weights]
print("Trainable layers:")
for name, params in trainable:
    print(f"  {name:<30} {params:>10,} params")


## 6. Train DME Head

In [ ]:
best_weights = os.path.join(OUTPUT_DIR, "dme_finetuned.weights.h5")

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_dme_risk_accuracy", patience=5,
        restore_best_weights=True, mode="max", verbose=1),
    tf.keras.callbacks.ModelCheckpoint(
        best_weights, monitor="val_dme_risk_accuracy",
        save_best_only=True, save_weights_only=True, mode="max", verbose=1),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3,
        min_lr=1e-7, verbose=1),
    tf.keras.callbacks.TensorBoard(
        log_dir=os.path.join(OUTPUT_DIR, "tensorboard_logs")),
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    class_weight=class_weights,
    callbacks=callbacks,
    verbose=1,
)


## 7. Save Training History

In [ ]:
hist_path = os.path.join(OUTPUT_DIR, "training_history.json")
hist_dict = {k: [float(v) for v in vals]
             for k, vals in history.history.items()}
with open(hist_path, "w") as f:
    json.dump(hist_dict, f, indent=2)
print(f"History saved → {hist_path}")


## 8. Plot Training History

In [ ]:
from evaluate import plot_training_curves
matplotlib.use("Agg")
plot_training_curves(hist_path,
                     output_path=os.path.join(OUTPUT_DIR, "training_curves.png"))

img_arr = plt.imread(os.path.join(OUTPUT_DIR, "training_curves.png"))
plt.figure(figsize=(12, 4))
plt.imshow(img_arr)
plt.axis("off")
plt.title("Training Curves")
plt.tight_layout()
plt.show()


## 9. Evaluate on Validation Set

In [ ]:
from evaluate import evaluate, predict_dme, compute_metrics

metrics = evaluate(
    model=model,
    dataset=val_ds,
    output_dir=OUTPUT_DIR,
    history_path=hist_path,
)

print(f"
Accuracy : {metrics['accuracy']:.4f}")
print(f"Macro F1 : {metrics['macro_f1']:.4f}")
print(f"QWK      : {metrics['qwk']:.4f}")


## 10. Confusion Matrix

In [ ]:
cm_path = os.path.join(OUTPUT_DIR, "confusion_matrix.png")
img_arr = plt.imread(cm_path)
plt.figure(figsize=(14, 5))
plt.imshow(img_arr)
plt.axis("off")
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()


## 11. Final Report

In [ ]:
report_path = os.path.join(OUTPUT_DIR, "metrics_report.txt")
with open(report_path) as f:
    print(f.read())
